# Notebook 1 · Seeing Linear Programs

**Algorithms & Geometry of Linear Programming — IBS Summer School, August 2026**

Companion notebook to Lecture 1, *Introduction to Linear Programming*. Estimated time: **35 minutes**.

| Part | Topic | Lecture section | Time |
|---|---|---|---|
| A | The geometric view: sliding the objective over a polygon | The LP model | ≈ 10 min |
| B | The algebraic view in three dimensions: vertices as tight, invertible subsystems | Polyhedra | ≈ 15 min |
| C | The certificate game: duality and the Farkas lemma | Duality | ≈ 10 min |

**How to work — no programming is required.** Run the setup cell, then proceed from top to bottom. You will interact in three ways only:

1. **Move sliders** in the interactive cells, and **rotate** the three-dimensional figures of Part B with the mouse.
2. **Answer checkpoints** by replacing the `...` after an `=` sign with a number (or a short list of numbers), then running the *check* cell below it for instant feedback.
3. **Edit the data** of the random linear program of Section B.3 and re-run the cell to see the effect.

Throughout, a linear program (LP) is written in the *geometric form* of the lecture,
$$\max\ c^\top x \quad \text{subject to} \quad Ax \le b,$$
Parts A and C work in the plane ($n = 2$); Part B moves to $n = 3$, where the feasible region is a solid that can be rotated with the mouse.


In [ ]:
#@title Setup — all algorithms and drawing utilities (run me) { display-mode: "form" }
import numpy as np
import matplotlib.pyplot as plt
from itertools import combinations
from scipy.optimize import linprog

np.set_printoptions(precision=4, suppress=True)


def solve_lp(A, b, c):
    """Solve max c^T x subject to A x <= b.  Returns (status, x, value)."""
    A = np.asarray(A, float); b = np.asarray(b, float); c = np.asarray(c, float)
    res = linprog(-c, A_ub=A, b_ub=b,
                  bounds=[(None, None)] * A.shape[1], method="highs")
    if res.status == 0:
        return "optimal", res.x, float(c @ res.x)
    return {2: "infeasible", 3: "unbounded"}.get(res.status, "error"), None, None


def tight_set(A, b, x, tol=1e-6):
    """Indices of the constraints tight at x (a_i^T x = b_i)."""
    A = np.asarray(A, float); b = np.asarray(b, float)
    return [i for i in range(len(b))
            if abs(A[i] @ x - b[i]) <= tol * (1 + abs(b[i]))]


def enumerate_vertices(A, b, tol=1e-7):
    """All vertices of {x : A x <= b}, found by solving every square
    subsystem A_I x = b_I with invertible A_I and keeping the feasible ones."""
    A = np.asarray(A, float); b = np.asarray(b, float)
    m, n = A.shape
    cand = []
    for I in combinations(range(m), n):
        A_I = A[list(I)]
        if np.linalg.matrix_rank(A_I) < n:
            continue
        x = np.linalg.solve(A_I, b[list(I)])
        if np.all(A @ x <= b + tol * (1 + np.abs(b))):
            cand.append(x)
    if not cand:
        return np.zeros((0, n))
    return np.unique(np.round(np.array(cand), 8), axis=0)


def enumerate_vertices_report(A, b, tol=1e-7):
    """Run the enumerator and report what happened to every subsystem."""
    A = np.asarray(A, float); b = np.asarray(b, float)
    m, n = A.shape
    kept, infeas, sing = [], 0, 0
    for I in combinations(range(m), n):
        A_I = A[list(I)]
        if np.linalg.matrix_rank(A_I) < n:
            sing += 1
            continue
        x = np.linalg.solve(A_I, b[list(I)])
        if np.all(A @ x <= b + tol * (1 + np.abs(b))):
            kept.append(x)
        else:
            infeas += 1
    V = (np.unique(np.round(np.array(kept), 8), axis=0)
         if kept else np.zeros((0, n)))
    total = len(list(combinations(range(m), n)))
    print(f"examined all C({m},{n}) = {total} square subsystems A_I x = b_I:")
    print(f"  {len(V)} gave vertices (feasible solutions),")
    print(f"  {infeas} gave infeasible solutions (corners outside the region),")
    print(f"  {sing} were singular (parallel constraint normals).")
    return V


def _display_vertices(A, b, box, tol=1e-7):
    A = np.asarray(A, float); b = np.asarray(b, float)
    Bx = np.array([[1., 0.], [-1., 0.], [0., 1.], [0., -1.]])
    dx = np.array([box[1], -box[0], box[3], -box[2]], float)
    A2, b2 = np.vstack([A, Bx]), np.concatenate([b, dx])
    V = []
    for i, j in combinations(range(len(b2)), 2):
        M = A2[[i, j]]
        if abs(np.linalg.det(M)) < tol:
            continue
        v = np.linalg.solve(M, b2[[i, j]])
        if np.all(A2 @ v <= b2 + 1e-6):
            V.append(v)
    if not V:
        return np.zeros((0, 2))
    V = np.unique(np.round(np.array(V), 8), axis=0)
    ctr = V.mean(axis=0)
    ang = np.arctan2(V[:, 1] - ctr[1], V[:, 0] - ctr[0])
    return V[np.argsort(ang)]


def draw_lp(A, b, c=None, labels=None, xlim=(-1, 5), ylim=(-1, 4), ax=None,
            show_normals=True, legend_loc="upper right"):
    """The feasible region of A x <= b; with c, also the objective arrow,
    dashed level lines, and the optimum.  Small arrows point into each
    feasible halfplane."""
    A = np.asarray(A, float); b = np.asarray(b, float)
    if ax is None:
        _, ax = plt.subplots(figsize=(6.2, 6.2))
    span = max(xlim[1] - xlim[0], ylim[1] - ylim[0])
    L = 2.0 * span
    ctr = np.array([(xlim[0] + xlim[1]) / 2, (ylim[0] + ylim[1]) / 2])
    V = _display_vertices(A, b, (xlim[0], xlim[1], ylim[0], ylim[1]))
    if len(V) >= 3:
        ax.fill(V[:, 0], V[:, 1], color="#9ecae1", alpha=0.55, zorder=1)
    for i, (a, bi) in enumerate(zip(A, b)):
        na = np.linalg.norm(a)
        if na < 1e-12:
            continue
        p0 = a * bi / na**2
        d = np.array([-a[1], a[0]]) / na
        seg = np.array([p0 - L * d, p0 + L * d])
        lab = labels[i] if labels is not None and i < len(labels) else None
        (h,) = ax.plot(seg[:, 0], seg[:, 1], lw=1.5, label=lab, zorder=2)
        if show_normals:
            foot = ctr - (a / na) * ((a / na) @ ctr - bi / na)
            tip = foot - 0.06 * span * a / na
            ax.annotate("", xy=tip, xytext=foot, zorder=2,
                        arrowprops=dict(arrowstyle="->", color=h.get_color()))
    if c is not None:
        c = np.asarray(c, float)
        status, xopt, val = solve_lp(A, b, c)
        if status == "optimal":
            nc = np.linalg.norm(c)
            delta = 0.15 * span * nc
            for k in range(3):
                t = val - k * delta
                p0 = c * t / nc**2
                d = np.array([-c[1], c[0]]) / nc
                seg = np.array([p0 - L * d, p0 + L * d])
                ax.plot(seg[:, 0], seg[:, 1], "--", color="gray",
                        lw=2.0 if k == 0 else 0.9, zorder=3)
            ax.annotate("", xy=xopt + 0.14 * span * c / nc, xytext=xopt,
                        arrowprops=dict(arrowstyle="-|>", lw=2, color="crimson"),
                        zorder=5)
            ax.plot(*xopt, "o", ms=9, mfc="crimson", mec="black", zorder=6)
            ax.annotate(r"$c$", xopt + 0.16 * span * c / nc, color="crimson",
                        fontsize=12, zorder=5)
    if labels is not None:
        ax.legend(loc=legend_loc, fontsize=8, framealpha=0.9)
    ax.set_xlim(xlim); ax.set_ylim(ylim); ax.set_aspect("equal")
    ax.axhline(0, color="black", lw=0.5, zorder=0)
    ax.axvline(0, color="black", lw=0.5, zorder=0)
    return ax


def lp_widget(A, b, labels=None, xlim=(-1, 5), ylim=(-1, 4)):
    """Rotate the objective c = (cos t, sin t) with a slider and watch the
    optimum, its value and its tight set change."""
    A = np.asarray(A, float); b = np.asarray(b, float)

    def show(angle=0.60):
        c = np.array([np.cos(angle), np.sin(angle)])
        _, ax = plt.subplots(figsize=(6.2, 6.2))
        draw_lp(A, b, c=c, labels=labels, xlim=xlim, ylim=ylim, ax=ax)
        status, x, val = solve_lp(A, b, c)
        if status == "optimal":
            I = tight_set(A, b, x)
            ax.set_title(f"angle = {angle:.2f}    c = ({c[0]:+.2f}, {c[1]:+.2f})\n"
                         f"x* = ({x[0]:.2f}, {x[1]:.2f})    value = {val:.2f}"
                         f"    tight set I = {I}", fontsize=10)
        else:
            ax.set_title(f"angle = {angle:.2f}    status: {status.upper()}",
                         fontsize=11)
        plt.show()

    try:
        from ipywidgets import interact, FloatSlider
        interact(show, angle=FloatSlider(value=0.60, min=0.0, max=2 * np.pi,
                                         step=0.01, description="angle of c",
                                         continuous_update=False))
    except Exception:
        print("(ipywidgets unavailable — showing four snapshots instead)")
        for th in (0.3, 0.79, 2.0, 3.6):
            show(th)


def verify_certificate(A, b, c, x, y, tol=1e-3):
    """Audit an optimality certificate (x, y) line by line."""
    A = np.asarray(A, float); b = np.asarray(b, float)
    c = np.asarray(c, float); y = np.asarray(y, float)
    checks = {
        "x is feasible:   A x <= b        ": bool(np.all(A @ x <= b + tol)),
        "y is a valid combination:  y >= 0": bool(np.all(y >= -tol)),
        "y derives the objective: A^T y = c": bool(np.allclose(A.T @ y, c, atol=tol)),
        "values agree:    b^T y = c^T x   ": bool(abs(b @ y - c @ x) <= tol),
    }
    for name, ok in checks.items():
        print(("PASS  " if ok else "FAIL  ") + name)
    return all(checks.values())


# Canonical data of the production LP (used by the certificate game of Part C).
A_PROD = np.array([[1., 1.], [1., 3.], [-1., 0.], [0., -1.]])
B_PROD = np.array([4., 6., 0., 0.])
C_PROD = np.array([2., 3.])
X_STAR = np.array([3., 1.])
PROD_LABELS = ["wood", "labor", "x1 >= 0", "x2 >= 0"]


def certificate_game():
    """Find dual multipliers proving that x* = (3, 1) is optimal for the
    production LP.  One slider per constraint; the certificate closes when
    every condition turns to OK."""
    def show(y_wood=0.0, y_labor=0.0, y_x1=0.0, y_x2=0.0):
        y = np.array([y_wood, y_labor, y_x1, y_x2])
        Aty = A_PROD.T @ y
        bty = float(B_PROD @ y)
        val = float(C_PROD @ X_STAR)
        _, ax = plt.subplots(figsize=(6.0, 6.0))
        draw_lp(A_PROD, B_PROD, c=C_PROD, labels=PROD_LABELS,
                xlim=(-0.5, 5.6), ylim=(-0.5, 4.6), ax=ax,
                legend_loc="lower left")
        # target: the dashed arrow c, drawn at x*
        ax.annotate("", xy=X_STAR + C_PROD, xytext=X_STAR, zorder=6,
                    arrowprops=dict(arrowstyle="-|>", lw=2.5, color="gray",
                                    linestyle="--"))
        ax.annotate("target $c$", X_STAR + 0.5 * C_PROD + [-0.95, 0.15],
                    color="gray", fontsize=10)
        # your combination: the arrows y_i a_i, tip to tail from x*
        pos = X_STAR.copy()
        for yi, a in zip(y, A_PROD):
            if yi > 1e-9:
                ax.annotate("", xy=pos + yi * a, xytext=pos, zorder=7,
                            arrowprops=dict(arrowstyle="-|>", lw=2.5,
                                            color="seagreen"))
                pos = pos + yi * a
        ax.set_title("stack the green arrows $y_i a_i$ onto the dashed target $c$")
        plt.show()
        ok1 = np.allclose(Aty, C_PROD, atol=1e-3)
        ok2 = bool(np.all(y >= -1e-9))
        ok3 = abs(bty - val) <= 1e-3
        mk = lambda o: "OK  " if o else "--  "
        print(f"{mk(ok1)} A^T y = ({Aty[0]:6.2f}, {Aty[1]:6.2f})     "
              f"target  c = ({C_PROD[0]:.2f}, {C_PROD[1]:.2f})")
        print(f"{mk(ok2)} y >= 0")
        print(f"{mk(ok3)} b^T y = {bty:6.2f}                 "
              f"target  c^T x* = {val:.2f}")
        if ok1 and ok2 and ok3:
            print("\nCERTIFICATE COMPLETE:  c^T x = y^T A x <= y^T b = c^T x*")
            print("for every feasible x — hence x* = (3, 1) is optimal.")
    try:
        from ipywidgets import interact, FloatSlider
        sl = dict(min=0.0, max=3.0, step=0.05, continuous_update=False)
        interact(show, y_wood=FloatSlider(value=0.0, **sl),
                 y_labor=FloatSlider(value=0.0, **sl),
                 y_x1=FloatSlider(value=0.0, **sl),
                 y_x2=FloatSlider(value=0.0, **sl))
    except Exception:
        print("(ipywidgets unavailable — showing the starting position)")
        show()


A_INF = np.array([[1., 1.], [-1., 0.], [0., -1.]])
B_INF = np.array([1., -2., 0.])
INF_LABELS = ["x1 + x2 <= 1", "x1 >= 2", "x2 >= 0"]


def farkas_game():
    """Find weights y >= 0 that combine the three (contradictory) constraints
    into the impossibility 0 <= -1."""
    def show(y_1=0.0, y_2=0.0, y_3=0.0):
        y = np.array([y_1, y_2, y_3])
        Aty = A_INF.T @ y
        bty = float(B_INF @ y)
        _, ax = plt.subplots(figsize=(5.4, 5.4))
        draw_lp(A_INF, B_INF, labels=INF_LABELS, xlim=(-1, 4),
                ylim=(-1.5, 3), ax=ax)
        ax.set_title("an empty intersection: no region is shaded")
        plt.show()
        ok1 = np.allclose(Aty, 0, atol=1e-3)
        ok2 = bool(np.all(y >= -1e-9))
        ok3 = abs(bty + 1.0) <= 1e-3
        mk = lambda o: "OK  " if o else "--  "
        print(f"{mk(ok1)} A^T y = ({Aty[0]:6.2f}, {Aty[1]:6.2f})     "
              f"target  (0, 0)   — x drops out entirely")
        print(f"{mk(ok2)} y >= 0")
        print(f"{mk(ok3)} b^T y = {bty:6.2f}                 target  -1")
        if ok1 and ok2 and ok3:
            print("\nCERTIFICATE COMPLETE: summing y_i * (a_i^T x <= b_i)")
            print("derives 0 <= -1 — a one-line proof of infeasibility.")
    try:
        from ipywidgets import interact, FloatSlider
        sl = dict(min=0.0, max=3.0, step=0.05, continuous_update=False)
        interact(show, y_1=FloatSlider(value=0.0, **sl),
                 y_2=FloatSlider(value=0.0, **sl),
                 y_3=FloatSlider(value=0.0, **sl))
    except Exception:
        print("(ipywidgets unavailable — showing the starting position)")
        show()


# ------------------------------------------------------------- 3D utilities

def _bounded_directions(A, b):
    """True if {x : A x <= b} is bounded (all coordinate LPs are finite)."""
    for e in np.eye(A.shape[1]):
        for s in (1.0, -1.0):
            if solve_lp(A, b, s * e)[0] == "unbounded":
                return False
    return True


def _edges_3d(A, b, V, tol=1e-6):
    """Index pairs (i, j) such that the segment [V[i], V[j]] is an edge of
    the polytope: the constraints tight at both endpoints have rank n - 1."""
    A = np.asarray(A, float)
    E = []
    for i, j in combinations(range(len(V)), 2):
        Ti, Tj = tight_set(A, b, V[i], tol), tight_set(A, b, V[j], tol)
        common = [k for k in Ti if k in Tj]
        if len(common) >= 2 and np.linalg.matrix_rank(A[common]) == 2:
            E.append((i, j))
    return E


def show_lp_3d(A, b, c=None, title=None, show_tight_sets=True):
    """Interactive 3D view of P = {x : A x <= b} in R^3 (drag to rotate).
    Hovering over a vertex shows its coordinates and (if show_tight_sets)
    its tight set I.  With an objective c, the optimal vertex is highlighted
    in red and the objective direction is drawn as an arrow based there."""
    A = np.asarray(A, float); b = np.asarray(b, float)
    V = enumerate_vertices(A, b)
    status = None
    if c is not None:
        status, x_opt, val = solve_lp(A, b, c)
        if status == "infeasible":
            print("This system is INFEASIBLE: no point satisfies all constraints.")
            return
        if status == "unbounded":
            print("The LP is UNBOUNDED: the objective increases without limit "
                  "along a recession direction of the region.")
    if len(V) == 0:
        if solve_lp(A, b, np.zeros(A.shape[1]))[0] == "infeasible":
            print("This system is INFEASIBLE: no point satisfies all constraints.")
        else:
            print("The region is nonempty but has no vertices "
                  "(it is unbounded and contains a line).")
        return
    bounded = _bounded_directions(A, b)
    if not bounded:
        print("Note: the region is UNBOUNDED — the figure shows only its "
              "vertices and bounded edges; the region extends beyond them.")
    E = _edges_3d(A, b, V)

    hover = []
    for v in V:
        txt = f"x = ({v[0]:.2f}, {v[1]:.2f}, {v[2]:.2f})"
        if show_tight_sets:
            txt += f"<br>I = {tight_set(A, b, v)}"
        hover.append(txt)

    head = title or ""
    if c is not None and status == "optimal":
        I_opt = tight_set(A, b, x_opt)
        line = (f"x* = ({x_opt[0]:.2f}, {x_opt[1]:.2f}, {x_opt[2]:.2f}),  "
                f"value = {val:.2f},  tight set I = {I_opt}")
        head = f"{head}<br>{line}" if head else line

    try:
        import plotly.graph_objects as go
        data = []
        if bounded and len(V) >= 4:
            try:
                from scipy.spatial import ConvexHull
                hull = ConvexHull(V, qhull_options="QJ")
                data.append(go.Mesh3d(
                    x=V[:, 0], y=V[:, 1], z=V[:, 2],
                    i=hull.simplices[:, 0], j=hull.simplices[:, 1],
                    k=hull.simplices[:, 2],
                    color="#9ecae1", opacity=0.35, flatshading=True,
                    hoverinfo="skip"))
            except Exception:
                pass
        ex, ey, ez = [], [], []
        for i, j in E:
            ex += [V[i, 0], V[j, 0], None]
            ey += [V[i, 1], V[j, 1], None]
            ez += [V[i, 2], V[j, 2], None]
        data.append(go.Scatter3d(x=ex, y=ey, z=ez, mode="lines",
                                 line=dict(color="#404040", width=4),
                                 hoverinfo="skip"))
        data.append(go.Scatter3d(
            x=V[:, 0], y=V[:, 1], z=V[:, 2], mode="markers",
            marker=dict(size=5, color="black"),
            text=hover, hoverinfo="text"))
        if c is not None and status == "optimal":
            data.append(go.Scatter3d(
                x=[x_opt[0]], y=[x_opt[1]], z=[x_opt[2]], mode="markers",
                marker=dict(size=9, color="crimson", symbol="diamond"),
                text=[hover[int(np.argmin(np.linalg.norm(V - x_opt, axis=1)))]],
                hoverinfo="text"))
            cn = np.asarray(c, float)
            cn = cn / np.linalg.norm(cn)
            span = float(np.max(V.max(axis=0) - V.min(axis=0)))
            tip = x_opt + 0.35 * span * cn
            data.append(go.Scatter3d(
                x=[x_opt[0], tip[0]], y=[x_opt[1], tip[1]],
                z=[x_opt[2], tip[2]], mode="lines",
                line=dict(color="crimson", width=6), hoverinfo="skip"))
            data.append(go.Cone(
                x=[tip[0]], y=[tip[1]], z=[tip[2]],
                u=[cn[0]], v=[cn[1]], w=[cn[2]],
                sizemode="absolute", sizeref=0.12 * span,
                anchor="tail", showscale=False,
                colorscale=[[0, "crimson"], [1, "crimson"]],
                hoverinfo="skip"))
        fig = go.Figure(data=data)
        fig.update_layout(
            title=dict(text=head, font=dict(size=13)),
            scene=dict(xaxis_title="x1", yaxis_title="x2", zaxis_title="x3",
                       aspectmode="data"),
            margin=dict(l=0, r=0, t=60, b=0), showlegend=False,
            width=650, height=550)
        fig.show()
    except ImportError:
        # Static fallback if plotly is unavailable.
        from mpl_toolkits.mplot3d.art3d import Line3DCollection
        fig = plt.figure(figsize=(6.5, 6.0))
        ax = fig.add_subplot(projection="3d")
        segs = [[V[i], V[j]] for i, j in E]
        ax.add_collection3d(Line3DCollection(segs, colors="#404040", lw=1.5))
        ax.scatter(V[:, 0], V[:, 1], V[:, 2], color="black", s=25)
        if show_tight_sets:
            for v in V:
                ax.text(v[0], v[1], v[2], f" I={tight_set(A, b, v)}", fontsize=7)
        if c is not None and status == "optimal":
            ax.scatter(*x_opt, color="crimson", s=80, zorder=5)
        ax.set_xlabel("x1"); ax.set_ylabel("x2"); ax.set_zlabel("x3")
        ax.set_title(head.replace("<br>", "\n"), fontsize=10)
        plt.show()


def print_instance_3d(A, b, c):
    """Print an LP instance as code that can be pasted into a cell and edited."""
    A = np.asarray(A, float); b = np.asarray(b, float); c = np.asarray(c, float)
    rows = []
    for i, a in enumerate(A):
        entries = ", ".join(f"{v:3.0f}." for v in a)
        prefix = "A_my = np.array([[" if i == 0 else "                 ["
        suffix = "]])" if i == len(A) - 1 else "],"
        rows.append(f"{prefix}{entries}{suffix}")
    print("\n".join(rows))
    print("b_my = np.array([" + ", ".join(f"{v:3.0f}." for v in b) + "])")
    print("c_my = np.array([" + ", ".join(f"{v:3.0f}." for v in c) + "])")


def random_lp_3d(seed=0, m=6, verbose=True):
    """Generate a random LP  max c^T x  s.t.  A x <= b  in R^3 with m
    constraints and small integer data.  The right-hand side is drawn
    nonnegative, so x = 0 satisfies every constraint and the region is
    feasible by construction.  Instances are resampled until the region is
    in addition a bounded polytope, every vertex is nondegenerate, and the
    optimal vertex is unique.  Prints the instance in editable form and
    returns (A, b, c)."""
    rng = np.random.default_rng(seed)
    for _ in range(20000):
        A = rng.integers(-3, 4, size=(m, 3)).astype(float)
        if np.any(np.all(A == 0, axis=1)):
            continue
        b = rng.integers(0, 7, size=m).astype(float)   # b >= 0: x = 0 is feasible
        c = rng.integers(-3, 4, size=3).astype(float)
        if np.all(c == 0):
            continue
        if not _bounded_directions(A, b):
            continue
        V = enumerate_vertices(A, b)
        if len(V) < 6:
            continue
        if np.max(np.abs(V)) > 8:
            continue                       # very elongated -- draws poorly
        if any(len(tight_set(A, b, v)) != 3 for v in V):
            continue                       # degenerate vertex — resample
        vals = np.sort(V @ c)
        if len(vals) >= 2 and vals[-1] - vals[-2] < 0.5:
            continue                       # near-tie — optimum not clearly unique
        if verbose:
            print(f"# Random instance (seed={seed}) — paste into the next "
                  "cell to edit it:\n")
            print_instance_3d(A, b, c)
        return A, b, c
    raise RuntimeError("no suitable instance found — try another seed")

print("Setup complete.")


## Part A — The geometric view (≈ 10 min)

Each constraint $a_i^\top x \le b_i$ is a **halfspace** whose boundary is the line $a_i^\top x = b_i$; the feasible region $P = \{x : Ax \le b\}$ is the intersection of the halfspaces. To maximize $c^\top x$, slide the level line $c^\top x = t$ (dashed) in the direction of $c$ until it is about to leave $P$: the last point (or edge) of contact is the optimal face.

The example below is the polygon from the lecture's references (Matoušek–Gärtner, Chapter 1):
$$\max\ c^\top x \quad \text{s.t.} \quad x_2 - x_1 \le 1,\quad x_1 + 6x_2 \le 15,\quad 4x_1 - x_2 \le 10,\quad x_1, x_2 \ge 0.$$

Run the next cell and move the slider, which rotates the objective $c = (\cos\theta, \sin\theta)$. The title reports the optimum $x^*$, its value, and — the algebraic shadow of the picture — the **tight set** $I$ of constraints active at $x^*$ (rows numbered $0$–$4$ in the order of the legend).


In [ ]:
# The Matousek--Gartner polygon, in the form A x <= b (rows numbered 0..4).
A_mg = np.array([[-1.0,  1.0],     # 0:  x2 - x1 <= 1
                 [ 1.0,  6.0],     # 1:  x1 + 6 x2 <= 15
                 [ 4.0, -1.0],     # 2:  4 x1 - x2 <= 10
                 [-1.0,  0.0],     # 3:  x1 >= 0
                 [ 0.0, -1.0]])    # 4:  x2 >= 0
b_mg = np.array([1.0, 15.0, 10.0, 0.0, 0.0])
mg_labels = ["0: x2 - x1 <= 1", "1: x1 + 6 x2 <= 15", "2: 4 x1 - x2 <= 10",
             "3: x1 >= 0", "4: x2 >= 0"]

lp_widget(A_mg, b_mg, labels=mg_labels, xlim=(-1, 5), ylim=(-1, 4))


### Checkpoints A.1 and A.2

Use the slider above to answer, then record your angles below and run the check cell.

- **A.1.** Find an angle at which the tight set is $I = [1, 2]$. *(Each vertex is optimal for a whole interval of angles — the cone spanned by its tight constraint normals, the optimality certificate of the lecture. Any angle in the right interval will do.)*
- **A.2.** Find an angle at which an **entire edge** is optimal, i.e., the dashed level line comes to rest *on* an edge rather than at a single corner. *(Hint: this happens precisely when $c$ points along a constraint normal $a_i = (a_{i1}, a_{i2})$; such an angle can be written `np.arctan2(a_i2, a_i1)`. Two decimal places from the slider are also accepted.)*


In [ ]:
my_angle      = ...     # A.1: an angle whose optimum has tight set [1, 2]
my_edge_angle = ...     # A.2: an angle at which an entire edge is optimal


In [ ]:
# --- Check for A.1 and A.2 ---------------------------------------------------
assert my_angle is not Ellipsis, "A.1: replace ... by a number first"
assert my_edge_angle is not Ellipsis, "A.2: replace ... by a number first"

c1 = np.array([np.cos(my_angle), np.sin(my_angle)])
_, x1, _ = solve_lp(A_mg, b_mg, c1)
I = tight_set(A_mg, b_mg, x1)
assert I == [1, 2], (f"A.1: at angle {my_angle:.2f} the optimum is "
                     f"({x1[0]:.2f}, {x1[1]:.2f}) with tight set {I}, "
                     "not [1, 2] — keep exploring")
print(f"A.1 correct: at angle {my_angle:.2f} the optimum is "
      f"({x1[0]:.2f}, {x1[1]:.2f}), where constraints 1 and 2 meet.")

c2 = np.array([np.cos(my_edge_angle), np.sin(my_edge_angle)])
V = enumerate_vertices(A_mg, b_mg)
vals = np.sort(V @ c2)[::-1]
rel_gap = (vals[0] - vals[1]) / max(vals[0] - vals[-1], 1e-9)
assert rel_gap < 0.015, ("A.2: at your angle a single vertex is strictly "
                         "best — align c with a constraint normal")
print(f"A.2 correct: at angle {my_edge_angle:.2f} two vertices are (near-)tied,")
print("so the whole edge between them is optimal — the level line rests on it.")


## Part B — Vertices, algebraically (≈ 15 min)

The lecture characterizes a **vertex** of $P = \{x \in \mathbb{R}^n : Ax \le b\}$ algebraically: $x$ is a vertex if and only if it is the *unique* solution of a square subsystem $A_I x = b_I$ with $|I| = n$ and $A_I$ invertible, and $x$ satisfies all remaining constraints. This yields a (very inefficient, but instructive) procedure: try all $\binom{m}{n}$ subsystems, solve each, and keep the feasible solutions.

Nothing in this characterization is special to the plane, so we now move to $n = 3$, where vertices, edges and facets can all be seen at once. The example is the box $0 \le x_1 \le 2$, $\ 0 \le x_2 \le 2$, $\ x_3 \ge 0$, truncated by the constraint $x_1 + x_2 + x_3 \le 3$ (rows numbered $0$–$5$):

$$-x_1 \le 0, \qquad -x_2 \le 0, \qquad -x_3 \le 0, \qquad x_1 \le 2, \qquad x_2 \le 2, \qquad x_1 + x_2 + x_3 \le 3.$$

With $m = 6$ constraints in $n = 3$ variables the procedure will examine $\binom{6}{3} = 20$ subsystems. Every subsystem meets one of three fates: it yields a **vertex**; it yields an **infeasible** intersection point (three boundary planes meeting outside the region); or it is **singular** (no unique intersection point).

### Checkpoint B.1 — predict before running

Run the next cell, rotate the solid with the mouse until you have seen every side, and count its corners. *How many of the 20 subsystems will yield vertices?* Record your prediction, then run the two cells that follow.

In [ ]:
# A three-dimensional polytope: the box 0 <= x1 <= 2, 0 <= x2 <= 2, x3 >= 0,
# truncated by the constraint x1 + x2 + x3 <= 3  (rows numbered 0..5).
A_3d = np.array([[-1.,  0.,  0.],   # 0:  x1 >= 0
                 [ 0., -1.,  0.],   # 1:  x2 >= 0
                 [ 0.,  0., -1.],   # 2:  x3 >= 0
                 [ 1.,  0.,  0.],   # 3:  x1 <= 2
                 [ 0.,  1.,  0.],   # 4:  x2 <= 2
                 [ 1.,  1.,  1.]])  # 5:  x1 + x2 + x3 <= 3
b_3d = np.array([0., 0., 0., 2., 2., 3.])

show_lp_3d(A_3d, b_3d, show_tight_sets=False,
           title="drag to rotate; hover over a corner for its coordinates")

In [ ]:
predicted_vertex_count = ...     # B.1: how many of the 20 subsystems are vertices?

In [ ]:
V = enumerate_vertices_report(A_3d, b_3d)

# The same polytope, redrawn with the objective c = (3, 1, 2): the optimal
# vertex is highlighted in red.  Hover over each corner to read the tight
# set I that names it algebraically.
show_lp_3d(A_3d, b_3d, c=[3., 1., 2.],
           title="every vertex = a tight, invertible 3x3 subsystem A_I x = b_I")

In [ ]:
# --- Check for B.1 -----------------------------------------------------------
assert predicted_vertex_count is not Ellipsis, "replace ... by a number first"
assert predicted_vertex_count == len(V), \
    (f"the enumerator found {len(V)} vertices, you predicted "
     f"{predicted_vertex_count} — rotate the figure and count again")
print(f"B.1 correct: {len(V)} of the 20 subsystems are vertices — one per")
print("corner of the polytope.  Geometry counted them; algebra confirmed them.")

### Checkpoint B.2 — read the figure

The figure above attaches to each vertex its tight set — the *algebraic name* of the geometric corner. Which tight set $I$ names the optimal vertex $(2, 0, 1)$? Enter the three constraint indices below (in any order) and run the check.

In [ ]:
tight_set_of_2_0_1 = [..., ..., ...]    # B.2: the three indices naming the vertex (2, 0, 1)

In [ ]:
# --- Check for B.2 -----------------------------------------------------------
assert Ellipsis not in tight_set_of_2_0_1, "replace all three ... by indices first"
answer = sorted(int(i) for i in tight_set_of_2_0_1)
truth = tight_set(A_3d, b_3d, np.array([2.0, 0.0, 1.0]))
assert answer == truth, (f"the tight set at (2, 0, 1) is {truth}, "
                         f"you entered {answer} — hover over the vertex again")
print("B.2 correct: (2, 0, 1) is the unique solution of the 3x3 system")
print("  -x2           = 0   (constraint 1)")
print("   x1           = 2   (constraint 3)")
print("   x1 + x2 + x3 = 3   (constraint 5)  — its algebraic name is I = [1, 3, 5].")

### B.3 — Random polytopes and their optima

The enumeration procedure and the three-dimensional view apply to any instance, not only to hand-built examples. The next cell generates a random linear program $\max\ c^\top x$ s.t. $Ax \le b$ with $m = 6$ constraints in $\mathbb{R}^3$ and small integer data, prints its data, and draws it with the optimum highlighted. The right-hand side $b$ is drawn nonnegative, so the origin satisfies every constraint and the instance is feasible by construction (a device worth remembering: any $b \ge 0$ makes $x = 0$ feasible); the generator additionally resamples until the region is bounded and the optimal vertex is unique. Change the seed for a fresh instance; each seed produces a different polytope.

The second cell contains one instance written out in full, ready to edit: rotate the objective by changing `c_my`, translate a facet by changing an entry of `b_my`, or tilt a facet by editing a row of `A_my`, then re-run the cell to see how the polytope and its optimum respond. If an edit makes the region empty or unbounded, the cell reports this instead — both phenomena from the lecture's solution trichotomy are worth producing on purpose.

In [ ]:
A_rnd, b_rnd, c_rnd = random_lp_3d(seed=1)   # change the seed for a fresh instance
show_lp_3d(A_rnd, b_rnd, c_rnd)

In [ ]:
# An instance written out in full — edit it and re-run.  Suggestions:
#   * change the objective c_my and watch the optimal vertex move,
#   * increase or decrease an entry of b_my (this translates a facet),
#   * edit a row of A_my (this tilts a facet),
#   * negate a row of A_my together with its entry of b_my (this reverses
#     the halfspace, and may make the region empty or unbounded).
A_my = np.array([[ 2.,  1.,  2.],
                 [ 1., -1., -2.],
                 [ 0.,  1., -1.],
                 [-2., -3., -3.],
                 [-2., -1., -2.],
                 [-2., -1.,  2.]])
b_my = np.array([ 2.,  6.,  4.,  0.,  6.,  6.])
c_my = np.array([ 2., -2.,  2.])

show_lp_3d(A_my, b_my, c_my)

**Why not always enumerate?** For $m = 50$ constraints in $n = 20$ variables there are $\binom{50}{20} \approx 4.7 \cdot 10^{13}$ subsystems — enumeration is hopeless. The simplex method of Lecture 2 (Notebook 2) instead *walks* from vertex to neighboring vertex, exchanging one element of the tight set $I$ per step. The optimum of a bounded LP is attained at a vertex, as the lecture guarantees, so walking on vertices suffices.


## Part C — The certificate game (≈ 10 min)

Consider the production LP from the lecture: a workshop makes chairs (profit $2$, using $1$ wood and $1$ labor) and tables (profit $3$, using $1$ wood and $3$ labor), with $4$ wood and $6$ labor available,
$$\max\ 2x_1 + 3x_2 \quad \text{s.t.} \quad x_1 + x_2 \le 4,\quad x_1 + 3x_2 \le 6,\quad x_1, x_2 \ge 0,$$
whose optimal solution is $x^* = (3, 1)$, of value $9$. How can one *prove* that $x^*$ is optimal, without checking every feasible point? The lecture's answer is a **dual certificate**: a vector $y \ge 0$ with $A^\top y = c$ and $b^\top y = c^\top x^*$. Indeed, for every feasible $x$,
$$c^\top x = (A^\top y)^\top x = y^\top (Ax) \le y^\top b = c^\top x^*,$$
so three lines of arithmetic replace an infinite search. Strong duality asserts that such a $y$ always exists at an optimum.

**The game.** Below you have one slider per constraint of the production LP. Each slider value $y_i$ contributes an arrow $y_i a_i$ (green, stacked tip-to-tail from $x^*$); your task is to make the green chain land exactly on the dashed target arrow $c$ — that is, $y_1 a_1 + \dots + y_4 a_4 = c$ — using **nonnegative** multipliers. The scoreboard below the figure tracks all three conditions and announces when the certificate is complete.

*Geometric hint from Part A / the lecture: at the optimum, $c$ lies in the cone of the **tight** constraint normals. Which two constraints are tight at $(3,1)$?*


In [ ]:
certificate_game()


### Checkpoint C.1 — record your certificate

Enter the four slider values that completed the certificate, then run the check, which audits your $y$ line by line.


In [ ]:
my_y = [..., ..., ..., ...]      # C.1: (y_wood, y_labor, y_x1, y_x2)


In [ ]:
# --- Check for C.1 -----------------------------------------------------------
assert Ellipsis not in my_y, "replace every ... by a slider value first"
ok = verify_certificate(A_PROD, B_PROD, C_PROD, X_STAR, my_y)
assert ok, "one of the four conditions above failed — return to the game"
print("\nC.1 correct: y = (1.5, 0.5, 0, 0).  Interpretation: at the optimum,")
print("one extra unit of wood is worth 1.5 and one of labor 0.5 — the")
print("'shadow prices' of the resources.  And indeed 1.5*4 + 0.5*6 = 9.")


### Certifying infeasibility: the Farkas game

The system below demands $x_1 + x_2 \le 1$, $\ x_1 \ge 2$, $\ x_2 \ge 0$ — plainly impossible. The Farkas lemma of the lecture promises a certificate of this fact: a vector $y \ge 0$ with $A^\top y = 0$ and $b^\top y = -1$. Summing the constraints $a_i^\top x \le b_i$ with weights $y_i$ then derives the contradiction $0 \le -1$.

**Round two.** Same rules, new target: make $A^\top y = (0, 0)$ — so that $x$ drops out of the combination entirely — while $b^\top y = -1$.


In [ ]:
farkas_game()


In [ ]:
my_farkas_y = [..., ..., ...]    # C.2: the three slider values (y_1, y_2, y_3)


In [ ]:
# --- Check for C.2 -----------------------------------------------------------
assert Ellipsis not in my_farkas_y, "replace every ... by a slider value first"
y = np.asarray(my_farkas_y, float)
assert np.all(y >= -1e-9), "the multipliers must be nonnegative"
assert np.allclose(A_INF.T @ y, 0, atol=1e-3), "A^T y should vanish"
assert abs(B_INF @ y + 1) < 1e-3, "b^T y should equal -1"
print("C.2 correct: with weights y =", np.round(y, 3))
print("summing  y_i * (a_i^T x <= b_i)  yields  0 <= -1: a one-line proof")
print("of infeasibility.  Whether A x <= b is feasible or not, a short")
print("certificate of the correct answer always exists (Farkas lemma).")


## Where this leads

- Part A's "slide the level line until it leaves $P$" becomes an algorithm once we know how to move between corners: the **simplex method** (Lecture 2, Notebook 2).
- Part B's tight sets $I$ are precisely the **bases** that the simplex method exchanges one element at a time; a pivot is a controlled change of $I$.
- Part C's certificates return throughout the course: as the optimality test of the simplex method, and as the dual half of the central path in Lectures 4–5.

**Bonus (optional).** The widget below removes two constraints from the polygon of Part A, leaving an *unbounded* feasible region (the shading is truncated by the plot window; the region continues beyond it). For some angles the LP still has an optimum; for others the status flips to `UNBOUNDED` — the solution trichotomy of the lecture. Which set of directions $c$ makes the LP unbounded? *(Answer: those making a positive inner product with some recession direction of the region — compare with the recession cone $\{r : Ar \le 0\}$.)*


In [ ]:
A_unb = np.array([[-1.0,  1.0],     # x2 - x1 <= 1
                  [-1.0,  0.0],     # x1 >= 0
                  [ 0.0, -1.0]])    # x2 >= 0
b_unb = np.array([1.0, 0.0, 0.0])

lp_widget(A_unb, b_unb, labels=["x2 - x1 <= 1", "x1 >= 0", "x2 >= 0"],
          xlim=(-1, 6), ylim=(-1, 5))
